In [220]:
import pandas as pd
import os

In [221]:
INPUT_FILES_DIR = r"dataset\output\temporal"
TOTAL_CLUSTERS = len(os.listdir(INPUT_FILES_DIR))

print(TOTAL_CLUSTERS)

5


In [222]:
INPUT_PATH = os.path.join(INPUT_FILES_DIR,"cluster1_temporalSort.csv")

In [223]:
df = pd.read_csv(INPUT_PATH)
df = df.reset_index(drop=True)
df['sent_id'] = df.index
df['sentence'] = df['sentence'].astype(str).str.strip()

In [224]:
print(f"total sentences: {len(df)}")
df.head()

total sentences: 12


,article_id,sentence,pub_date,hdb_cluster,formatted_date,sent_id
0,184,this is a preventative step we re choosing to ...,"Fri, 25 Mar 2022 10:33:43 GMT",1,2022-03-25 10:33:43+00:00,0
1,728,anti-spam laws bite spammer hard the net s sel...,"Fri, 01 Apr 2022 20:58:12 GMT",1,2022-04-01 20:58:12+00:00,1
2,728,for its part optinrealbig describes itself as ...,"Fri, 01 Apr 2022 20:58:12 GMT",1,2022-04-01 20:58:12+00:00,2
3,728,in a statement announcing the desire to seek b...,"Fri, 01 Apr 2022 20:58:12 GMT",1,2022-04-01 20:58:12+00:00,3
4,728,in its chapter 11 filing optinrealbig claimed ...,"Fri, 01 Apr 2022 20:58:12 GMT",1,2022-04-01 20:58:12+00:00,4


In [225]:
UNIQUE_ARTICLES = df.article_id.unique()
print(UNIQUE_ARTICLES)

[184 728]


In [226]:
article = df[df['article_id']==UNIQUE_ARTICLES[1]].copy().reset_index(drop=True)
article

,article_id,sentence,pub_date,hdb_cluster,formatted_date,sent_id
0,728,anti-spam laws bite spammer hard the net s sel...,"Fri, 01 Apr 2022 20:58:12 GMT",1,2022-04-01 20:58:12+00:00,1
1,728,for its part optinrealbig describes itself as ...,"Fri, 01 Apr 2022 20:58:12 GMT",1,2022-04-01 20:58:12+00:00,2
2,728,in a statement announcing the desire to seek b...,"Fri, 01 Apr 2022 20:58:12 GMT",1,2022-04-01 20:58:12+00:00,3
3,728,in its chapter 11 filing optinrealbig claimed ...,"Fri, 01 Apr 2022 20:58:12 GMT",1,2022-04-01 20:58:12+00:00,4
4,728,listed as the third biggest spammer in the wor...,"Fri, 01 Apr 2022 20:58:12 GMT",1,2022-04-01 20:58:12+00:00,5
5,728,microsoft is seeking millions in dollars in da...,"Fri, 01 Apr 2022 20:58:12 GMT",1,2022-04-01 20:58:12+00:00,6
6,728,mr richter settled the attorney general case i...,"Fri, 01 Apr 2022 20:58:12 GMT",1,2022-04-01 20:58:12+00:00,7
7,728,scott richter the man behind optinrealbig.com ...,"Fri, 01 Apr 2022 20:58:12 GMT",1,2022-04-01 20:58:12+00:00,8
8,728,the company said filing for chapter 11 would h...,"Fri, 01 Apr 2022 20:58:12 GMT",1,2022-04-01 20:58:12+00:00,9
9,728,the lawsuit was brought by microsoft and new y...,"Fri, 01 Apr 2022 20:58:12 GMT",1,2022-04-01 20:58:12+00:00,10


In [227]:
import re
from datetime import datetime, timedelta

OUTPUT_PATH = "temporal_sorted_sentences.csv"

In [228]:
SENT_COL = "sentence"
ARTICLE_COL = "article_id"
PUBDATE_COL = "pub_date"
SENT_ORDER_COL = "sent_id"

In [229]:
df = article.copy().reset_index(drop=True)
print("Loaded:", INPUT_PATH)
print("Rows:", len(df))
print("Columns:", list(df.columns))

Loaded: dataset\output\temporal\cluster1_temporalSort.csv
Rows: 11
Columns: ['article_id', 'sentence', 'pub_date', 'hdb_cluster', 'formatted_date', 'sent_id']


In [230]:
# Patterns for absolute dates and years
YEAR_RE = re.compile(r"\b(19|20)\d{2}\b")
MONTH_NAME_RE = re.compile(r"\b(?:jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|jun(?:e)?|jul(?:y)?|aug(?:ust)?|sep(?:t(?:ember)?)?|oct(?:ober)?|nov(?:ember)?|dec(?:ember)?)\b", re.IGNORECASE)
DAY_MONTH_YEAR_RE = re.compile(r"\b(\d{1,2})[\/\-\s](\d{1,2})[\/\-\s]((?:19|20)\d{2})\b")  # dd-mm-yyyy or dd/mm/yyyy
MONTH_YEAR_RE = re.compile(r"\b(" + "|".join([m for m in ["January","February","March","April","May","June","July","August","September","October","November","December"]]) + r")\s+((?:19|20)\d{2})\b", re.IGNORECASE)
# Relative time keywords
RELATIVE_KEYWORDS = {
    r"\b(yesterday)\b": lambda ref: ref - timedelta(days=1),
    r"\b(today|currently|at present|now)\b": lambda ref: ref,
    r"\b(tomorrow)\b": lambda ref: ref + timedelta(days=1),
    r"\b(last year|previous year|the year before)\b": lambda ref: ref.replace(year=ref.year-1),
    r"\b(next year|following year)\b": lambda ref: ref.replace(year=ref.year+1),
    r"\b(last month)\b": lambda ref: (ref - pd.DateOffset(months=1)).to_pydatetime(),
    r"\b(next month)\b": lambda ref: (ref + pd.DateOffset(months=1)).to_pydatetime(),
    r"\b(\d+)\s+years?\s+ago\b": None,  # handled separately
    r"\b(\d+)\s+months?\s+ago\b": None,
    r"\b(earlier this year|earlier this month)\b": lambda ref: ref,  # weak signal -> keep ref
    r"\b(recently|recent)\b": lambda ref: ref,
    r"\b(announced|said|reported|stated|revealed|claimed)\b": None  # event verbs: no date but useful as signal
}

# Sequence markers and their ordering score (lower -> earlier)
SEQUENCE_MARKERS = {
    r"\b(first|initially|to begin with|at first)\b": 1,
    r"\b(then|next|afterwards|after that|subsequently)\b": 2,
    r"\b(later|following this|following that)\b": 3,
    r"\b(finally|lastly|in conclusion)\b": 4,
    r"\b(previously|earlier)\b": 0.5,
    r"\b(currently|now)\b": 2.5
}

In [231]:
def extract_year(text):
    m = YEAR_RE.search(text)
    if m:
        return int(m.group(0))
    return None

def extract_day_month_year(text):
    m = DAY_MONTH_YEAR_RE.search(text)
    if m:
        d, mo, y = m.groups()
        try:
            return datetime(int(y), int(mo), int(d))
        except:
            return None
    return None

def extract_month_year(text):
    m = MONTH_YEAR_RE.search(text)
    if m:
        month_name, year = m.groups()
        try:
            dt = datetime.strptime(f"{month_name} {year}", "%B %Y")
            return dt
        except:
            try:
                dt = datetime.strptime(f"{month_name} {year}", "%b %Y")
                return dt
            except:
                return None
    return None

def extract_relative(text, ref_date):
    text_low = text.lower()
    # exact functions
    for pat, func in RELATIVE_KEYWORDS.items():
        if re.search(pat, text_low):
            if func is None:
                # handle numeric "X years ago" or "X months ago"
                m = re.search(r"(\d+)\s+years?\s+ago", text_low)
                if m:
                    years = int(m.group(1))
                    try:
                        return ref_date.replace(year=ref_date.year - years)
                    except:
                        # fallback: use days approx
                        return ref_date - timedelta(days=years*365)
                m2 = re.search(r"(\d+)\s+months?\s+ago", text_low)
                if m2:
                    months = int(m2.group(1))
                    return (ref_date - pd.DateOffset(months=months)).to_pydatetime()
                # no concrete mapping
                return None
            else:
                try:
                    return func(ref_date)
                except Exception:
                    return None
    return None

def seq_marker_score(text):
    text_low = text.lower()
    scores = []
    for pat, score in SEQUENCE_MARKERS.items():
        if re.search(pat, text_low):
            scores.append(score)
    if scores:
        # pick earliest (min)
        return min(scores)
    return None

# Apply extraction per sentence
def analyze_row(row):
    text = str(row[SENT_COL])
    pub = row["pub_date"]
    if pd.isna(pub):
        # fallback to today's date for relative normalization
        pub = datetime.now()
    # 1. explicit dd-mm-yyyy
    dm = extract_day_month_year(text)
    if dm is not None:
        return {"date": dm, "source": "explicit_dm"}
    # 2. month year
    my = extract_month_year(text)
    if my is not None:
        return {"date": my, "source": "month_year"}
    # 3. explicit year
    y = extract_year(text)
    if y is not None:
        try:
            dt = datetime(y, 6, 15)  # approximate mid-year when only year is present
            return {"date": dt, "source": "year_only"}
        except:
            pass
    # 4. relative phrases
    rel = extract_relative(text, pub)
    if rel is not None:
        return {"date": rel, "source": "relative"}
    # 5. sequence markers
    seq = seq_marker_score(text)
    if seq is not None:
        # map sequence score to a pseudo-date: use pub_date minus (5 - seq) days as approximate ordering
        pseudo = pub + timedelta(days=0)  # baseline at pub date
        # earlier markers -> subtract days; later markers -> add days slightly
        offset_days = int((seq - 2) * 2)  # seq 1 -> -2 days, seq 4 -> +4 days
        pseudo = pseudo + timedelta(days=offset_days)
        return {"date": pseudo, "source": "sequence_marker", "seq_score": seq}
    # 6. no signal -> None
    return {"date": None, "source": None}


In [232]:
# Run analysis
meta = df.apply(analyze_row, axis=1, result_type="expand")
df["_extracted_dt"] = pd.to_datetime(meta["date"], errors="coerce")
df["_date_source"] = meta["source"]
df["_seq_score"] = meta.get("seq_score")

In [233]:
df.head()

,article_id,sentence,pub_date,hdb_cluster,formatted_date,sent_id,_extracted_dt,_date_source,_seq_score
0,728,anti-spam laws bite spammer hard the net s sel...,"Fri, 01 Apr 2022 20:58:12 GMT",1,2022-04-01 20:58:12+00:00,1,NaT,None,None
1,728,for its part optinrealbig describes itself as ...,"Fri, 01 Apr 2022 20:58:12 GMT",1,2022-04-01 20:58:12+00:00,2,NaT,None,None
2,728,in a statement announcing the desire to seek b...,"Fri, 01 Apr 2022 20:58:12 GMT",1,2022-04-01 20:58:12+00:00,3,NaT,None,None
3,728,in its chapter 11 filing optinrealbig claimed ...,"Fri, 01 Apr 2022 20:58:12 GMT",1,2022-04-01 20:58:12+00:00,4,NaT,None,None
4,728,listed as the third biggest spammer in the wor...,"Fri, 01 Apr 2022 20:58:12 GMT",1,2022-04-01 20:58:12+00:00,5,2003-12-01,month_year,None


In [ ]:
df['pub_date'].iloc[0]

TypeError: function missing required argument 'year' (pos 1)

In [235]:
# Compute temporal_score: if extracted date -> timestamp, else fallback to pub_date +/- sent order
def compute_temporal_score(row):
    if pd.notna(row["_extracted_dt"]):
        # use timestamp as score
        return row["_extracted_dt"].timestamp()
    # no extracted date: use pub_date as baseline plus small offset by sentence order
    base = row["pub_date"]
    if pd.isna(base):
        base = datetime.now()
    # offset by original sentence order to keep stability
    offset = int(row[SENT_ORDER_COL]) * 60  # 60 seconds per sent to preserve original order
    return base.timestamp() + offset

df["_temporal_score"] = df.apply(compute_temporal_score, axis=1)

# Sort within each article by temporal score (ascending = earlier first)
df_sorted = df.sort_values([ARTICLE_COL, "_temporal_score"])

# Save result
df_sorted.to_csv(OUTPUT_PATH, index=False)
print("Saved sorted sentences to:", OUTPUT_PATH)

# Provide feedback summary
print("\nSummary of date source counts:")
print(df_sorted["_date_source"].value_counts(dropna=False))

AttributeError: 'str' object has no attribute 'timestamp'